# Customer support LLM judge add-on

Offline analysis of 200 preserved predictions. Judge feedback remains separate from reference labels. Run all cells with Python 3.10+; no API calls or external packages. See METHOD.md for rubric choices and official documentation sources. The implementing AI inspected the pilot; independent participant adjudication remains open.

In [1]:
from pathlib import Path
import sys, json, csv
ROOT = Path.cwd()
if not (ROOT / 'inputs').exists():
    ROOT = ROOT / 'p4' / 'llm-judge'
assert (ROOT / 'inputs').exists(), 'Run from llm-judge or the Week-4 workspace'
sys.path.insert(0, str(ROOT.resolve()))
from analysis import analyze
summary = analyze()
print('Completed assessments:', sum(summary[v]['assessments'] for v in ('baseline','focused')))
print('Model:', json.loads((ROOT/'config.json').read_text())['model'])

Completed assessments: 200
Model: gpt-5.4-mini-2026-03-17


## Method

Each request contains the same frozen system rubric and only query plus predicted_category. No version, ground truth, human approval, experiment name, existing score or note is sent. Structured feedback includes score, correctness, rationale, ambiguity, proposed intent and alternatives. Valid pilot assessments are counted once in the final 200. Refusals, truncation, invalid schema and inconsistent fields are failures requiring retry or explicit unresolved status.

The declared policy sends non-delivery refund requests to order_status and damaged-goods refund requests to product_issue. Missing refund money after returns and unwanted-order cancellations go to refund_request. This overlaps the focused prompt policy and does not independently validate the policy choice. Ambiguity can coexist with a definite rubric score.

In [2]:
print('Measure                         Baseline Focused')
for label,key in [('Original-label Exact Match','original_exact_match'),('Reviewed-label Exact Match','reviewed_exact_match'),('LLM rubric passes','judge_pass'),('Ambiguity flags','ambiguity_flags'),('Proposed intent vs original','judge_vs_original_label'),('Proposed intent vs reviewed','judge_vs_reviewed_label'),('Score agrees with original EM','score_vs_original_em'),('Score agrees with reviewed EM','score_vs_reviewed_em')]:
    print(f'{label:32} {summary["baseline"][key]:3}/100 {summary["focused"][key]:3}/100')

Measure                         Baseline Focused
Original-label Exact Match        92/100  94/100
Reviewed-label Exact Match        93/100  93/100
LLM rubric passes                 92/100  93/100
Ambiguity flags                   10/100  12/100
Proposed intent vs original      100/100  99/100
Proposed intent vs reviewed       99/100  98/100
Score agrees with original EM    100/100  99/100
Score agrees with reviewed EM     99/100  98/100


## Paired changes and consistency

Differences use preserved predictions for the same tickets. The 86 unchanged predictions produce identical judge requests twice; t033 receives conflicting scores on these identical inputs. This is evidence of judge instability, with only two calls per input. The 14 changed predictions contribute +2 passes, while the t033 inconsistency contributes -1, producing the observed +1 total. No judgment is replaced to force agreement. Paraphrases share 25 seed families, so 100 tickets do not constitute 100 independent held-out trials.

In [3]:
print(json.dumps(summary['paired'], indent=2))
print('Per original reference category:')
print(json.dumps(summary['by_reference_intent'], indent=2))
print('Per participant failure category, joined only after judging:')
print(json.dumps(summary['by_review_failure_category'], indent=2))
print('Passes per family, four paraphrases each:')
print(json.dumps(summary['by_seed_family'], indent=2))

{
  "changes": {
    "UNCHANGED_PASS": 85,
    "WIN": 8,
    "REGRESSION": 7
  },
  "judge_delta_percentage_points": 1,
  "proposed_intent_inconsistency_ids": [
    "t033"
  ],
  "ambiguity_inconsistency_ids": [
    "t033",
    "t056",
    "t057",
    "t059"
  ],
  "identical_input_pairs": 86,
  "identical_input_score_disagreement_ids": [
    "t033"
  ],
  "identical_input_score_delta": -1,
  "changed_prediction_score_delta": 2
}
Per original reference category:
{
  "baseline": {
    "order_status": {
      "n": 20,
      "judge_pass": 16
    },
    "refund_request": {
      "n": 20,
      "judge_pass": 20
    },
    "product_issue": {
      "n": 20,
      "judge_pass": 16
    },
    "account_help": {
      "n": 20,
      "judge_pass": 20
    },
    "other": {
      "n": 20,
      "judge_pass": 20
    }
  },
  "focused": {
    "order_status": {
      "n": 20,
      "judge_pass": 20
    },
    "refund_request": {
      "n": 20,
      "judge_pass": 13
    },
    "product_issue": {
      

## Disagreements and human review

Judge disagreement is a review request, not permission to change a label. Participant approval means reviewed, not correct. t046 was changed by the participant from product_issue to refund_request; both Exact Match comparisons preserve that fact. The judge uses the explicit damaged-product rule, which may disagree with that human policy preference.

In [4]:
def load_csv(name):
    with (ROOT/'results'/name).open(encoding='utf-8',newline='') as stream:
        return list(csv.DictReader(stream))
disagreements = load_csv('disagreements.csv') if (ROOT/'results/disagreements.csv').exists() else []
print('Disagreement assessments:', len(disagreements))
for r in disagreements:
    print(r['id'], r['version'], 'prediction='+r['prediction'], 'judge='+r['proposed_intent'], 'original='+r['original_label'], 'reviewed='+r['reviewed_label'], 'score='+r['score'])
print('Tickets needing review:', ', '.join(summary['review_ticket_ids']))

Disagreement assessments: 3
t033 focused prediction=refund_request judge=account_help original=refund_request reviewed=refund_request score=0
t046 baseline prediction=refund_request judge=product_issue original=product_issue reviewed=refund_request score=0
t046 focused prediction=product_issue judge=product_issue original=product_issue reviewed=refund_request score=1
Tickets needing review: t004, t005, t006, t007, t033, t044, t045, t046, t047, t056, t057, t058, t059


## Meaningful examples

Examples cover policy wins, regressions, a reviewed-label disagreement and a routine pass. These were selected for routing mechanisms, not to establish model superiority.

In [5]:
rows=load_csv('assessments.csv')
for ticket in ['t004','t024','t028','t033','t046','t094']:
    for r in rows:
        if r['id']==ticket:
            print(f"{ticket} {r['version']}: prediction={r['prediction']}; score={r['score']}; proposed={r['proposed_intent']}; ambiguity={r['ambiguity']}")
            print(' Query:', r['query'])
            print(' Rationale:', r['rationale'])

t004 baseline: prediction=refund_request; score=0; proposed=order_status; ambiguity=True
 Query: I never received my order and I want my money back.
 Rationale: The customer states non-delivery as the underlying issue (“I never received my order”), which the policy routes to order_status before the requested refund remedy.
t004 focused: prediction=order_status; score=1; proposed=order_status; ambiguity=True
 Query: I never received my order and I want my money back.
 Rationale: The customer states they never received the order, which is a non-delivery issue routed to order_status, and the money-back request is secondary under the precedence rule.
t024 baseline: prediction=refund_request; score=1; proposed=refund_request; ambiguity=False
 Query: Where is my refund? I returned the item two weeks ago and still nothing on my card.
 Rationale: The customer is asking about a refund that has not yet appeared after returning the item, which is a missing refund-payment issue routed to refund_re

## Cost, failures and limits

Cost is estimated from returned token usage, including rejected responses with usage. Transport failures without usage have unknown billing. The judge can share model-family biases with the classifier, infer the wrong intent, anchor on predictions or inconsistently flag ambiguity. The pilot is not an independently human-labelled calibration set. No significance or real-world improvement claim follows from this reused synthetic sample. Neither the labels nor the original 93/100 versus 93/100 reviewed result is changed.

In [6]:
print(json.dumps(summary['usage'], indent=2))
print('Recorded failure types:', summary['failure_types'])
print('Rejected-response score changes:', summary['retry_sensitivity']['score_changes'])
print('Rejected-response proposed-intent changes:', summary['retry_sensitivity']['proposed_intent_changes'])
print('Pilot review:')
print((ROOT/'evidence/pilot_review.json').read_text())
print('Frozen input hashes verified by analyze().')

{
  "api_attempts": 254,
  "failed_attempts": 54,
  "prompt_tokens": 152206,
  "cached_prompt_tokens": 0,
  "completion_tokens_including_reasoning": 24222,
  "interrupted_unrecorded_requests_maximum": 4,
  "estimated_cost_usd": 0.2231535,
  "cost_note": "Calculated from returned usage and published rates; not an invoiced charge. Missing response usage and interrupted in-flight calls cannot be billed exactly here."
}
Recorded failure types: {'RuntimeError': 29, 'ValueError': 24, 'RemoteDisconnected': 1}
Rejected-response score changes: 0
Rejected-response proposed-intent changes: 0
Pilot review:
{
  "decision": "PROCEED",
  "reviewer": "Codex AI-assisted technical review; not participant approval",
  "reviewed_at": "2026-10-31T03:07:39.042899+00:00",
  "rubric_sha256": "a0f7babda76f86a7a229671e158e677b3db9dd909062b6092dd427d88e3339c3",
  "valid_assessments": 16,
  "failed_attempts": 8,
  "routing_check": "All 16 accepted rationales inspected against frozen policy: delivery t000; non-del